# Experimentos com Autoencoder para Sales Forecast

**Objetivo:** Avaliar se Autoencoders (AE) podem agregar valor preditivo ao modelo campeao de previsao de vendas semanais (LightGBM, MAE baseline = 1.4247).

## Hipoteses testadas

1. **AE embeddings como features estaticas** - compressao nao-linear do perfil temporal de cada serie (pdv, sku) adicionada como 8 features no LightGBM.
2. **AE embeddings como variavel de cluster** - agrupar series semelhantes e treinar modelos separados (ou adicionar cluster_id como feature categorica).

## Resultado sumario

| Abordagem | MAE val | delta vs baseline | Veredito |
|-----------|---------|-------------------|----------|
| Baseline (campeao)        | 1.4247 | -        | referencia |
| + AE naive (leaky)        | 1.7131 | +20.24%  | **pior** (vazamento) |
| + AE causal               | 1.4227 | -0.14%   | neutro (redundante) |
| k=3 global+cluster_id     | 1.5087 | +5.89%   | pior |
| k=5 global+cluster_id     | 1.4653 | +2.85%   | pior |
| k=8 global+cluster_id     | 1.4786 | +3.78%   | pior |
| k=3 per-cluster           | 1.5188 | +6.60%   | pior |
| k=5 per-cluster           | 1.5276 | +7.22%   | pior |
| k=8 per-cluster           | 1.5376 | +7.92%   | pior |

**Conclusao:** AE embeddings **nao agregam valor preditivo** ao LightGBM ja bem feature-engineered. Em configuracoes mal aplicadas, introduzem vazamento de dados e pioram significativamente o modelo.

## Dataset
- **5,685,189** linhas semanais agregadas por (ano, semana, pdv, sku, dimensoes categoricas)
- **709,667** series temporais unicas (pdv, sku)
- Train: semanas 1-47 | Val: semanas 48-52
- Modelo campeao: LightGBM com features lag/rolling/coef_variacao + 8 categoricas (MAE=1.4218 reportado, 1.4247 reproduzido)

## Experimento 1: AE embeddings como features estaticas

### Arquitetura do AE
- Input: matriz (709,667 series x 47 semanas) de `log1p(quantidade)` por serie
- StandardScaler -> MLPRegressor (47 -> 32 -> 8 -> 32 -> 47), ativacao relu, alpha=0.001
- Bottleneck (8 dims) extraido manualmente via forward pass: `Z2 = relu(relu(X*W1 + b1)*W2 + b2)`
- 8 dimensoes usadas como features `ae_emb_0..7` no LightGBM

### Duas variantes

#### Variante A: NAIVE (vazamento)
- Embedding computado com semanas **1-47 completas**
- Usado como feature para TODAS as linhas, inclusive treino (semanas 1-47)
- Para uma linha na semana 30, o embedding contem informacao das semanas 31-47 (futuro relativo a essa linha)

#### Variante B: CAUSAL (sem vazamento)
- Para cada semana w, embedding computado com mascara: apenas semanas 1..(w-1) visiveis
- Forward pass por semana (47 passes matriciais de 709k x 47)
- Linhas de validacao (semanas 48-52) usam embedding full 1-47 (legitimo, sem vazamento)

### Resultados

```
BASELINE                 MAE = 1.4247  (best_iter=1000)
+AE naive (leaky)         MAE = 1.7131  (best_iter=38)
+AE causal               MAE = 1.4227  (best_iter=1000)
```

### Analise

**Variante naive (+20.24%)**: classico vazamento de dados. Indicios:
- `best_iter` cai de ~1000 para 38 (modelo colapsa rapidamente no early stopping)
- Modelo aprende a depender de feature que carrega informacao do futuro durante treino
- Em validacao, a feature perde esse poder (nao ha futuro), gerando mismatch treino->val

**Variante causal (-0.14%)**: essencialmente **neutro**, dentro do ruido. Diagnostico:
- O LightGBM campeao ja captura a mesma informacao temporal via lag_4, lag_52, rolling_mean_4/12/52
- AE apenas comprime o que ja esta representado de forma explicita e causal
- Adicionar 8 features redundantes nao ajuda nem atrapalha significativamente

### Licao

**Embeddings derivados do historico temporal das series sao redundantes quando o modelo base ja possui features lag/rolling bem desenhadas. Usar a abordagem naive (sem mascara causal) introduz vazamento catastrofico.**

## Experimento 2: AE embeddings para clustering de series

### Hipotese

Series com perfis temporais semelhantes (compressao via AE) podem se beneficiar de:
- **Modelos por cluster** (K LightGBMs separados, um por cluster)
- **cluster_id como feature categorica** em modelo global unico

### Metodo

1. Embedding de perfil (causal, semana 47 = semanas 1-46) para 709,667 series
2. K-Means com k=3, 5, 8 sobre as 8 dimensoes do embedding
3. Para cada k, comparar:
   - Global + cluster_id feature (1 LightGBM, cluster_id como categorica)
   - Per-cluster (K LightGBMs separados, predicao = modelo do cluster da serie)
4. MAE final = media das predicoes por cluster (ponderada naturalmente pelo numero de linhas)

### Resultados

```
baseline                  MAE = 1.4247
k=3 global+cluster_id     MAE = 1.5087  (+5.89%)
k=3 per-cluster           MAE = 1.5188  (+6.60%)
k=5 global+cluster_id     MAE = 1.4653  (+2.85%)  <- melhor dos piores
k=5 per-cluster           MAE = 1.5276  (+7.22%)
k=8 global+cluster_id     MAE = 1.4786  (+3.78%)
k=8 per-cluster           MAE = 1.5376  (+7.92%)
```

Distribuicao dos tamanhos (k=5): 24,486 / 454,172 / 64,867 / 5,856 / 160,286 series

MAE por cluster em k=8 (per-cluster):
- Melhor: cluster 2 (MAE=0.76) - 172k series, vendas regulares
- Pior:  cluster 6 (MAE=8.91) - 1,849 series, lojas pequenas erraticas

### Analise

Por que nao ajuda:

1. **cluster_id adiciona ruido ao global** - o LightGBM ja possui `categoria_pdv`, `categoria`, `marca` etc. como features categoricas. cluster_id deriva do embedding temporal, que por sua vez e redundante com lags/rollings. E uma projecao de informacoes que o modelo ja tem aplicadas.

2. **Modelos per-cluster sofrem de fragmentacao**: dividir os dados em subconjuntos menores reduz a amostra treinavel por modelo. Clusters pequenos (ex: 1,849 series) produzem modelos fracos. Cluster MAE=8.91 em k=8 evidencia o problema.

3. **k=5 foi o melhor dos piores** porque produz clusters mais balanceados (k=3 e k=8 tem clusters muito pequenos e outros dominantes).

4. **Generalizacao**: o sinal compartilhado pelo embedding e uma reducao de dimensionalidade de informacoes ja presentes nas features. Clustering sobre features redundantes nao revela novos agrupamentos uteis para a previsao.

## Conclusao final

**AE embeddings nao agregam valor preditivo a um LightGBM bem feature-engineered** com features temporais explicitas (lags, rollings em varias janelas + features categoricas).

### Roadmap negativo/positivo

- Features AE causais -> neutras (sem ganho real, mas tecnicamente corretas)
- Features AE naive -> catastroficamente pior (vazamento - **cuidado ao usar embeddings temporais sem mascara causal**)
- Clustering via AE -> pior em todas as configuracoes testadas

### Onde AE ainda pode ser util (nao testado / para o futuro)

1. **Cold-start**: AE sobre **metadados categoricos** (categoria, marca, pdv) para derivar embedding por produto/loja. Usar embedding medio do mesmo par categoria+marca para prever vendas de series novas sem historico. (Diferente de AE temporal - usa atributos, nao serie temporal)
2. **Denoising de targets ruidosos**: treinar denoising AE para suavizar series erraticas antes de treinar o regressor
3. **Imputacao de lacunas**: AE para preencher semanas mudas com historico parcial
4. **Arquitetura hibrida end-to-end**: rede neural temporal (DeepAR/N-BEATS/TFT) que aprende embeddings joint com forecasting - maior esforco
5. **Ferramenta analitica**: clusterizar series para diagnostico comercial (identificar perfis problematicos, agrupar lojas similares)

### Reprodutibilidade

- Scripts de experimentacao: copiar `ae_valid.py`, `ae_valid2.py`, `ae_cluster.py` para `scripts/`
- Runtime tipico: ~11 min por experimento de feature, ~55 min para clustering completo (k=3,5,8)
- Memoria peak ~10 GB (sistema com 20+ GB disponiveis recomendado)
- Python 3.11 via `.venv311` na raiz do workspace